In [ ]:
import sys
import os
sys.path.append(os.path.realpath('../../'))

: 

In [ ]:
from tqdm.auto import tqdm
from typing import List
import re
from typing import List, Tuple, Dict
from data.dataset import GraphDataset, ReimburseGraphDataset, DataAugmentationLevel, DialogNode, NodeType
import nltk
from statistics import mean 

: 

In [ ]:
human_data_train = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../resources/")
human_data_test = ReimburseGraphDataset('en/reimburse/test_graph.json', 'en/reimburse/test_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../resources/")
generated_data_train_v1 = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/generated/train_answers.json', False, augmentation=DataAugmentationLevel.ARTIFICIAL_ONLY, augmentation_path="en/reimburse/generated/train_questions_v1.json", resource_dir="../../resources/")
generated_data_train_v2 = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/generated/train_answers.json', False, augmentation=DataAugmentationLevel.ARTIFICIAL_ONLY, augmentation_path="en/reimburse/generated/train_questions_v2.json", resource_dir="../../resources/")
generated_data_train_v3 = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/generated/train_answers.json', False, augmentation=DataAugmentationLevel.ARTIFICIAL_ONLY, augmentation_path="en/reimburse/generated/train_questions_v3.json", resource_dir="../../resources/")

: 

In [66]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
chencherry = SmoothingFunction()

def calculate_self_bleu(node: DialogNode, n_grams: int = 3) -> float:
    questions = set([q.text for q in node.questions])
    scores = []
    for hypothesis in questions:
        # take each generated sentence as hypothesis once and test against all other questions as references.
        references = questions.difference(set([hypothesis])) 
        scores.append(sentence_bleu(list(references), hypothesis, smoothing_function=chencherry.method1, weights=[1/n_grams for _ in range(n_grams)]))
    # take average as self-bleu
    return mean(scores)


In [ ]:
datasets = {
    "Human Train": human_data_train,
    # "Human Test": human_data_test,
    "Gen V1": generated_data_train_v1,
    "Gen V2": generated_data_train_v2,
    "Gen V3": generated_data_train_v3
}

: 

In [68]:
NGRAMS = [1,2,3,4,5]


ngram_scores = {}
for ngram in tqdm(NGRAMS):
    dataset_scores = {}
    for dataset_name in datasets:
        scores = []
        for node in datasets[dataset_name].nodes_by_type[NodeType.INFO]:
            if len(node.questions) <= 1:
                continue
            scores.append(calculate_self_bleu(node, ngram))
        dataset_scores[dataset_name] = mean(scores)
    ngram_scores[ngram] = dataset_scores

100%|██████████| 5/5 [00:11<00:00,  2.37s/it]


In [69]:
print("BLEU")
print(ngram_scores)


{1: {'Human Train': 0.7770562488537773,
  'Human Test': 0.6021721981334996,
  'Gen V1': 0.9533895014749563,
  'Gen V2': 0.9490782132882306,
  'Gen V3': 0.8517171771986467},
 2: {'Human Train': 0.6821749528525342,
  'Human Test': 0.480485469926561,
  'Gen V1': 0.9185073822750446,
  'Gen V2': 0.9049550760084472,
  'Gen V3': 0.776305989018358},
 3: {'Human Train': 0.600774251437181,
  'Human Test': 0.39023959821545506,
  'Gen V1': 0.873387712154815,
  'Gen V2': 0.8505019196330329,
  'Gen V3': 0.7096502411590758},
 4: {'Human Train': 0.5397821639946383,
  'Human Test': 0.3303580849716172,
  'Gen V1': 0.8340059382616397,
  'Gen V2': 0.8035380755031903,
  'Gen V3': 0.6597925198721522},
 5: {'Human Train': 0.49108430118472124,
  'Human Test': 0.29055057824595476,
  'Gen V1': 0.80045991481279,
  'Gen V2': 0.7634912301344376,
  'Gen V3': 0.6208092987870925}}

# Self-BLEURT

In [73]:
# Install BLEURT: pip install git+https://github.com/google-research/bleurt.git
# GET BLEURT MODEL: wget https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip .

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

: 

In [ ]:
from bleurt import score

: 

In [ ]:
checkpoint = "BLEURT-20"
scorer = score.BleurtScorer(checkpoint)

: 

In [ ]:
import itertools

def calculate_self_bleurt(node: DialogNode) -> float:
    questions = set([q.text for q in node.questions])
    scores = []
    # do pair-wise 
    # for reference, hypothesis in itertools.combinations(questions, 2):
    #     score = scorer.score(references=[reference], candidates=[hypothesis])
    #     scores.extend(score)
 
    # do all at once
    # print(mean(scores))
    references, hypotheses = zip(*itertools.combinations(questions, 2))
    scores = scorer.score(references=references, candidates=hypotheses)
    
    return mean(scores)


: 

In [42]:
print("\n\n\n")
print("BLEURT")
datasets = {
    "Human Train": human_data_train,
    # "Human Test": human_data_test,
    "Gen V1": generated_data_train_v1,
    "Gen V2": generated_data_train_v2,
    "Gen V3": generated_data_train_v3
}

NGRAMS = [1,2,3,4,5]


ngram_scores = {}
for ngram in tqdm(NGRAMS):
    dataset_scores = {}
    for dataset_name in datasets:
        scores = []
        for node in datasets[dataset_name].nodes_by_type[NodeType.INFO]:
            if len(node.questions) <= 1:
                continue
            scores.append(calculate_self_bleurt(node))
        dataset_scores[dataset_name] = mean(scores)
    ngram_scores[ngram] = dataset_scores
print(ngram_scores)

  0%|          | 0/5 [1:03:04<?, ?it/s]


KeyboardInterrupt: 